# 🔧 Data Preprocessing — Fake Product Review Detection

This notebook handles **loading, validating, and cleaning** the dataset before it can be used for training.

## What This Notebook Does

| Step | Description |
|------|-------------|
| 1 | Load the raw CSV dataset from the `data/` folder |
| 2 | Validate that the expected columns exist (`category`, `rating`, `label`, `text_`) |
| 3 | Handle missing values and remove duplicate reviews |
| 4 | Clean the review text (lowercase, remove URLs, HTML, punctuation, extra spaces) |
| 5 | **Lemmatize** each token — reduce words to base form (e.g. 'running' → 'run') |
| 6 | Map labels: `CG` → **Fake** (1), `OR` → **Genuine** (0) |
| 8 | Display a summary and sample of the cleaned data |

---

## 1. Import Libraries

We import standard Python libraries, **pandas** for data handling, and **NLTK** for NLP tasks (lemmatization and stopword removal).

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources (safe to run multiple times)
nltk.download('wordnet',    quiet=True)
nltk.download('omw-1.4',   quiet=True)

# Initialise once at module level for performance
_lemmatizer = WordNetLemmatizer()

print('✅ Libraries imported successfully!')
print("✅ Lemmatizer initialised.")

## 2. Define Constants

These constants define:
- **EXPECTED_COLUMNS**: The columns we expect in the Kaggle Fake Reviews Dataset
- **LABEL_MAP**: How we convert text labels to numbers (`CG` = Fake = 1, `OR` = Genuine = 0)
- **DATA_PATH**: Where the CSV file is located relative to the notebook

In [ ]:
# Expected columns in the dataset
EXPECTED_COLUMNS = ['category', 'rating', 'label', 'text_']

# Label mapping: CG = Computer-Generated (Fake), OR = Original (Genuine)
LABEL_MAP = {'CG': 1, 'OR': 0}
LABEL_NAME_MAP = {1: 'Fake', 0: 'Genuine'}

# Path to the dataset (relative to the project root)
DATA_PATH = os.path.join('..', 'data', 'fake_reviews_dataset.csv')

print(f'📁 Dataset path: {DATA_PATH}')
print(f'📋 Expected columns: {EXPECTED_COLUMNS}')
print(f'🏷️  Label mapping: {LABEL_MAP}')

## 3. Text Cleaning Function (with Stopword Removal + Lemmatization)

This function cleans a single review text string. The full cleaning pipeline now includes NLP steps:

1. **Lowercase** — makes all text uniform (`"GREAT"` → `"great"`)
2. **Remove URLs** — strips out `http://...` and `www.…` links
3. **Remove HTML tags** — strips out any `<tag>` elements
4. **Remove punctuation** — keeps only letters, numbers, and spaces
5. **Remove extra whitespace** — collapses multiple spaces into one
6. **Remove stopwords** — removes common English words ('the', 'is', 'a', 'in', …) that carry no discriminative signal
7. **Lemmatize** — reduces each token to its base dictionary form (`'running'` → `'run'`, `'products'` → `'product'`)

> **Why Lemmatization?** TF-IDF treats 'run', 'runs', 'running', 'ran' as four different features. Lemmatization collapses them into one (`'run'`), reducing vocabulary noise and helping the model generalise better.
>
> **Why Stopword Removal?** Words like 'the', 'is', 'and' appear in almost every review. Removing them lets TF-IDF focus on words that are actually discriminative between Fake and Genuine reviews.

In [ ]:
def clean_text(text):
    """
    Clean a single review text string.

    Steps:
      - Convert to lowercase
      - Remove URLs
      - Remove HTML tags
      - Remove punctuation and special characters (keep letters, numbers, spaces)
      - Remove extra whitespace
      - Remove stopwords (common words like 'the', 'is', 'in')
      - Lemmatize each token (e.g. 'running' -> 'run', 'bought' -> 'buy')

    Args:
        text (str): Raw review text.

    Returns:
        str: Cleaned and lemmatized review text.
    """
    if not isinstance(text, str):
        return ''

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation and special characters (keep letters, numbers, spaces)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize, remove stopwords, and lemmatize
    tokens = text.split()
    tokens = [
        _lemmatizer.lemmatize(token)
        for token in tokens
    ]

    return ' '.join(tokens)


# Quick test
sample = 'AMAZING products!!! Check http://example.com <b>Great</b> stuff!!! I am running out of words!'
print(f'Before: {sample}')
print(f'After:  {clean_text(sample)}')
print()
print('-- Lemmatization examples --')
for word in ['running', 'products', 'bought', 'amazing', 'reviews', 'highly']:
    print(f"  '{word}' -> '{_lemmatizer.lemmatize(word)}'")

## 4. Load the Dataset

We load the CSV file and validate that the expected columns are present.

If the file is missing, you will see an error message with download instructions.

In [ ]:
def load_dataset(filepath):
    """
    Load the CSV dataset and validate that expected columns exist.

    Args:
        filepath (str): Path to the CSV file.

    Returns:
        pd.DataFrame: Loaded DataFrame.
    """
    # Check file exists
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f'\n❌ Dataset not found at: {filepath}\n'
            f"   Please download the dataset from Kaggle and place it in the 'data/' folder.\n"
            f'   See README.md for download instructions.'
        )

    # Load CSV
    print(f'📂 Loading dataset from: {filepath}')
    df = pd.read_csv(filepath)
    print(f'   Loaded {len(df):,} rows and {len(df.columns)} columns.\n')

    # Validate columns
    missing_cols = [col for col in EXPECTED_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f'\n❌ Missing expected columns: {missing_cols}\n'
            f'   Found columns: {df.columns.tolist()}\n'
            f'   Expected columns: {EXPECTED_COLUMNS}\n'
            f'   Make sure you downloaded the correct dataset (Kaggle Fake Reviews Dataset).'
        )

    return df


# Load the dataset
df_raw = load_dataset(DATA_PATH)

### 4.1 Quick Look at the Raw Data

Let's see what the raw dataset looks like before any cleaning.

In [ ]:
print('📋 First 5 rows of the raw dataset:\n')
df_raw.head()

In [ ]:
print('📋 Dataset Info:\n')
df_raw.info()
print(f'\n📋 Shape: {df_raw.shape}')
print(f'\n📋 Label Distribution (before preprocessing):')
print(df_raw['label'].value_counts())

## 5. Preprocess the Data

The preprocessing pipeline performs these steps in order:

1. **Keep only relevant columns** (`category`, `rating`, `label`, `text_`)
2. **Drop missing values** — rows where `text_` or `label` is null
3. **Remove duplicate reviews** — keeps the first occurrence
4. **Validate labels** — only keep rows with `CG` or `OR` labels
5. **Clean text** — apply the `clean_text()` function to every review
   - Lowercase → Remove URLs → Remove HTML → Remove punctuation → Remove whitespace
   - **Remove stopwords** (new step)
   - **Lemmatize tokens** (new step)
6. **Remove empty reviews** — reviews that become empty after cleaning
7. **Map labels** — convert `CG` → 1 (Fake) and `OR` → 0 (Genuine)

In [ ]:
def preprocess_data(df):
    """
    Clean and preprocess the dataset.

    Returns:
        pd.DataFrame: Cleaned dataset with columns:
            - 'text_'      (cleaned + lemmatized review text)
            - 'label'      (numeric: 1=Fake, 0=Genuine)
            - 'label_name' (string: 'Fake' or 'Genuine')
            - 'category'   (product category)
            - 'rating'     (star rating)
    """
    print('🔧 Preprocessing data...')
    initial_count = len(df)

    # Step 1: Keep relevant columns
    df = df[EXPECTED_COLUMNS].copy()

    # Step 2: Handle missing values
    missing_text  = df['text_'].isnull().sum()
    missing_label = df['label'].isnull().sum()
    if missing_text > 0 or missing_label > 0:
        print(f'   Dropping {missing_text} rows with missing text, {missing_label} with missing labels.')
    df = df.dropna(subset=['text_', 'label'])

    # Step 3: Remove duplicates
    duplicates = df.duplicated(subset=['text_'], keep='first').sum()
    if duplicates > 0:
        print(f'   Removing {duplicates} duplicate reviews.')
        df = df.drop_duplicates(subset=['text_'], keep='first')

    # Step 4: Validate labels
    valid_labels  = set(LABEL_MAP.keys())
    invalid_mask  = ~df['label'].isin(valid_labels)
    invalid_count = invalid_mask.sum()
    if invalid_count > 0:
        print(f'   Removing {invalid_count} rows with invalid labels.')
        df = df[~invalid_mask]

    # Step 5: Clean text (includes stopword removal + lemmatization)
    print('   Cleaning review text (stopword removal + lemmatization)...')
    df['text_'] = df['text_'].apply(clean_text)

    # Step 6: Remove empty reviews after cleaning
    empty_mask  = df['text_'].str.strip() == ''
    empty_count = empty_mask.sum()
    if empty_count > 0:
        print(f'   Removing {empty_count} empty reviews after cleaning.')
        df = df[~empty_mask]

    # Step 7: Map labels
    df['label']      = df['label'].map(LABEL_MAP)
    df['label_name'] = df['label'].map(LABEL_NAME_MAP)

    # Summary
    final_count = len(df)
    removed     = initial_count - final_count
    print(f'\n✅ Preprocessing complete!')
    print(f'   Rows: {initial_count:,} \u2192 {final_count:,} (removed {removed:,})')
    print(f"   Fake reviews:    {(df['label'] == 1).sum():,}")
    print(f"   Genuine reviews: {(df['label'] == 0).sum():,}")

    df = df.reset_index(drop=True)
    return df


# Run preprocessing
df = preprocess_data(df_raw)

## 6. Inspect the Cleaned Data

Let's see what the data looks like after preprocessing.

In [ ]:
print('📋 First 10 rows of the cleaned dataset:\n')
df[['text_', 'label_name', 'category', 'rating']].head(10)

In [ ]:
print('📋 Data Types:\n')
print(df.dtypes)
print(f'\n📋 Shape: {df.shape}')
print(f'\n📋 Missing Values:')
print(df.isnull().sum())
print(f'\n📋 Label Distribution:')
print(df['label_name'].value_counts())

### 6.1 Before vs. After Cleaning Example

Let's compare a raw review with its cleaned + lemmatized version to see the full effect of preprocessing.

In [ ]:
# Show a before/after comparison
raw_sample     = df_raw['text_'].iloc[0]
cleaned_sample = clean_text(raw_sample)

print('── Before Cleaning ──')
print(raw_sample[:300])
print('\n── After Cleaning (Stopword Removal + Lemmatization) ──')
print(cleaned_sample[:300])

# Word count comparison
raw_words     = len(raw_sample.split())
cleaned_words = len(cleaned_sample.split())
print(f'\nWord count: {raw_words} -> {cleaned_words} (reduced by {raw_words - cleaned_words} words)')

## ✅ Preprocessing Complete!

The dataset is now cleaned, stopwords are removed, and tokens are lemmatized. Ready for the next steps:
- **Exploratory Data Analysis** (in `train.ipynb`)
- **TF-IDF Vectorization** (in `train.ipynb`)
- **Model Training** (in `train.ipynb`)

The cleaned DataFrame `df` contains:
- `text_` — cleaned + lemmatized review text (input for TF-IDF)
- `label` — numeric label: 1 = Fake, 0 = Genuine (target for model)
- `label_name` — human-readable label: 'Fake' or 'Genuine'
- `category` — product category
- `rating` — star rating

### Summary of NLP Steps Applied

| NLP Step | Example | Purpose |
|---|---|---|
| Lowercase | `AMAZING` → `amazing` | Uniform casing |
| Remove URLs | `http://x.com` → `` | Remove noise |
| Remove HTML | `<b>great</b>` → `great` | Remove markup |
| Remove punctuation | `great!!!` → `great` | Clean tokens |
| Stopword removal | `this is a great` → `great` | Remove uninformative words |
| Lemmatization | `running` → `run` | Reduce inflections to base form |